# 🧰 quant-kit — Vast.ai Professional Benchmark Suite

**Professional benchmarks on A100 GPU (~$2-4 per model).**  
Runs: MMLU Pro, AIME 2024, LiveCodeBench, HumanEval, MATH, HellaSwag (full)  
Results auto-upload to your HuggingFace repo.

### Setup on Vast.ai
1. Rent an instance: **A100 40GB** or **RTX 4090** (cheapest with 24GB VRAM)
2. Choose template: `pytorch/pytorch:2.3.0-cuda12.1-cudnn8-runtime`
3. Open Jupyter notebook, upload this file
4. Set your config below and Run All Cells

**Estimated cost:** ~\$2-4 for a full benchmark run on A100

In [ ]:
# ── CONFIG — Edit these ────────────────────────────────────────────────
HF_TOKEN    = "hf_your_token_here"             # Your HuggingFace token
HF_REPO     = "Dhptl/gemma-4-12b-it-GGUF"      # Your HuggingFace model repo
QUANT_TYPE  = "Q4_K_M"                          # Which quant to benchmark

# Select which pro benchmarks to run (each adds time + cost)
RUN_MMLU_PRO      = True    # ~90 min on A100 (12,032 questions)
RUN_AIME          = True    # ~20 min on A100 (30 questions)
RUN_HUMANEVAL     = True    # ~30 min on A100 (164 code problems)
RUN_MATH          = True    # ~45 min on A100 (5,000 problems)
RUN_LIVECODEBENCH = False   # Requires special setup, skip by default
# ──────────────────────────────────────────────────────────────────────

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────
import subprocess, sys, os

os.environ["HF_TOKEN"] = HF_TOKEN

print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "huggingface_hub", "lm-eval[api]", "psutil",
    "llama-cpp-python",
    "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu121"
], check=True)

# Download llama.cpp binaries
import urllib.request
from pathlib import Path

LLAMA_VERSION = "b5262"
url = f"https://github.com/ggerganov/llama.cpp/releases/download/{LLAMA_VERSION}/llama-{LLAMA_VERSION}-bin-ubuntu-x64.zip"
urllib.request.urlretrieve(url, "/tmp/llama.zip")
subprocess.run(["unzip", "-q", "-o", "/tmp/llama.zip", "-d", "/tmp/llama"])
subprocess.run(["chmod", "+x"] + [str(p) for p in Path("/tmp/llama").rglob("llama-*")], check=False)
llama_dir = next(Path("/tmp/llama").rglob("llama-bench")).parent
print(f"llama.cpp ready at {llama_dir}")

In [ ]:
# ── Download GGUF from HuggingFace ─────────────────────────────────────
from huggingface_hub import hf_hub_download
from pathlib import Path

model_name = HF_REPO.split("/")[1]
base_name  = model_name.replace("-GGUF", "")
gguf_file  = f"{base_name}-{QUANT_TYPE}.gguf"
model_path = f"/workspace/{gguf_file}"

print(f"Downloading {gguf_file}...")
hf_hub_download(repo_id=HF_REPO, filename=gguf_file,
                local_dir="/workspace", token=HF_TOKEN)
size_gb = Path(model_path).stat().st_size / 1e9
print(f"Ready: {gguf_file} ({size_gb:.2f} GB)")

In [ ]:
# ── Build task list & run lm-eval ──────────────────────────────────────
import subprocess, sys, json
from pathlib import Path

tasks = []
if RUN_MMLU_PRO:      tasks.append("mmlu_pro")
if RUN_AIME:          tasks.append("aime24")
if RUN_HUMANEVAL:     tasks.append("humaneval")
if RUN_MATH:          tasks.append("math_500")

eval_results = {}
results_dir  = Path("/workspace/eval_results")
results_dir.mkdir(exist_ok=True)

if tasks:
    print(f"Running: {', '.join(tasks)}")
    print("This may take 1-3 hours depending on tasks selected...")

    cmd = [
        sys.executable, "-m", "lm_eval",
        "--model", "gguf",
        "--model_args", f"pretrained={model_path}",
        "--tasks", ",".join(tasks),
        "--output_path", str(results_dir),
        "--batch_size", "auto",
        "--device", "cuda",
    ]
    subprocess.run(cmd, text=True, timeout=21600)

    for result_file in results_dir.glob("**/*.json"):
        if "results" in result_file.name:
            with open(result_file) as f:
                data = json.load(f)
            for task, metrics in data.get("results", {}).items():
                key_metric = (
                    metrics.get("exact_match,none") or
                    metrics.get("acc_norm,none") or
                    metrics.get("acc,none")
                )
                if key_metric is not None:
                    eval_results[task] = round(key_metric * 100, 2)
                    print(f"  {task}: {eval_results[task]}%")
            break

print("\nAll tasks done!")

In [ ]:
# ── Upload results to HuggingFace ──────────────────────────────────────
import json
from huggingface_hub import HfApi

output = {
    "model":      HF_REPO,
    "quant":      QUANT_TYPE,
    "platform":   "Vast.ai A100 GPU",
    "benchmarks": eval_results,
}

result_file = f"/workspace/vastai_results_{QUANT_TYPE}.json"
with open(result_file, "w") as f:
    json.dump(output, f, indent=2)

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=result_file,
    path_in_repo=f"vastai_results_{QUANT_TYPE}.json",
    repo_id=HF_REPO,
    repo_type="model",
    commit_message=f"Add Vast.ai professional benchmark results for {QUANT_TYPE}"
)

print(f"\n✅ Results uploaded to https://huggingface.co/{HF_REPO}")
print("\nNow run model_card.py locally to update your README with all benchmark data!")
print("  python model_card.py --model gemma-4-12b-it --original google/gemma-4-12b-it")
print()
print(json.dumps(output, indent=2))